# SLD Asymmetry Measurements and $\sin^2\theta_W^{\mathrm{eff}}$

**Hadronic $A_{LR}$, leptonic coupling asymmetries $A_e, A_\mu, A_\tau$, and a global fit of the effective weak mixing angle.**

This is **notebook 3 of 3**, the measurement layer. Inputs are the per-channel
parquet files produced by
[notebook 01](./SLD_01_DataPreparation.ipynb); kinematic sanity checks live
in [notebook 02](./SLD_02_ValidationPlots.ipynb). This notebook does **not**
depend on notebook 02.

### What this notebook does

1. **Hadronic $A_{LR}$** (§5) — left-right cross-section asymmetry from the
   $Z\to q\bar q$ sample, following the selection of the 2000
   high-precision SLD analysis ([hep-ex/0004026](https://arxiv.org/abs/hep-ex/0004026)).
2. **Leptonic coupling asymmetries** $A_e, A_\mu, A_\tau$ (§6) — left-right
   forward-backward asymmetries in the three dilepton channels, following the
   selection of the 2001 SLD leptonic-coupling analysis
   ([hep-ex/0010015](https://arxiv.org/abs/hep-ex/0010015)).
3. **Global $\sin^2\theta_W^{\mathrm{eff}}$ fit** (§7) — inverse-variance
   combination of the hadronic and leptonic determinations.
4. **Forest plot** (§8) — publication-quality visualisation of all four
   independent SLD determinations, the global combination, and the PDG 2025
   collider average.

### Scope (lifted from the paper)

All results in this notebook are intended as a *demonstration of internal
consistency* of the released dataset, not as a re-derivation of the published
SLD measurements. Uncertainties quoted include only statistical and
beam-polarisation contributions; other systematic uncertainties are not
included. See §9 for the two known sources of discrepancy from the published
values (ZFITTER pole correction for $A_{LR}$, and selection / extraction
differences for the leptonic channels).

## 1. Imports and configuration

This notebook reads the parquet files written by notebook 01 from
`$SLD_BASE/datasets/minidst_processed/`. If `SLD_BASE` is unset we fall back
to `./sld`. Match whatever you used when running notebook 01:

```bash
export SLD_BASE=/path/to/your/SLD
```

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import awkward as ak
import numpy as np
import pandas as pd

from sld_resurrect.kinematics import build_particles
from sld_resurrect.selector import EventSelector

SLD_BASE         = Path(os.environ.get("SLD_BASE", "./sld"))
SELECTED_DIR     = SLD_BASE / "datasets/minidst_processed"
PLOTS_DIR        = SLD_BASE / "analysis/plots"
MEASUREMENTS_DIR = SLD_BASE / "analysis/measurements"
for d in (PLOTS_DIR, MEASUREMENTS_DIR):
    os.makedirs(d, exist_ok=True)

if not SELECTED_DIR.is_dir():
    raise FileNotFoundError(
        f"Selected-event directory {SELECTED_DIR} does not exist. "
        f"Run notebook 01 (SLD_01_DataPreparation.ipynb) first, or set "
        f"SLD_BASE to a tree that already contains its outputs."
    )
print(f"SLD_BASE         = {SLD_BASE.resolve()}")
print(f"SELECTED_DIR     = {SELECTED_DIR}")
print(f"PLOTS_DIR        = {PLOTS_DIR}")
print(f"MEASUREMENTS_DIR = {MEASUREMENTS_DIR}")

## 2. Load the selected datasets

Each of the four channels is loaded from the parquet file written by
notebook 01. The full event record (all banks) is retained, so we can rebuild
particles, access the per-event polarisation `PHBM.pol`, and re-derive any
observable in this notebook.

In [ ]:
CHANNELS_USED: tuple[str, ...] = ("hadronic", "ee", "mumu", "tautau")


def load_selected(channel: str) -> ak.Array:
    """Load ``sld_{channel}.parquet`` from notebook 01."""
    path = SELECTED_DIR / f"sld_{channel}.parquet"
    arr = ak.from_parquet(str(path))
    print(f"  {channel:<10} {len(arr):>7,} events")
    return arr


print("Loaded selected samples:")
selected = {ch: load_selected(ch) for ch in CHANNELS_USED}
particles = {ch: build_particles(arr) for ch, arr in selected.items()}

## 3. Shared statistical helpers

Three primitives are used everywhere downstream:

* `inverse_variance_combine(values, errors)` — the inverse-variance-weighted
  combination of $N$ independent measurements with Gaussian errors,

  $$ \hat\mu \;=\; \frac{\sum_i w_i\, v_i}{\sum_i w_i}, \qquad
     \hat\sigma \;=\; \frac{1}{\sqrt{\sum_i w_i}}, \qquad
     w_i \;\equiv\; \sigma_i^{-2}. $$

  Every per-year-group combination, every cross-channel universality average,
  and the final hadronic-plus-leptonic global combination all reduce to a
  single call into this function.
* `measurement_pull(v1, e1, v2, e2)` — the standardised residual between two
  measurements, $(v_1 - v_2)/\sqrt{e_1^2 + e_2^2}$. Used as a consistency
  check between the hadronic and leptonic estimators.
* `asymmetry_from_counts(n_a, n_b)` — the binomial counting asymmetry
  $(N_A - N_B)/(N_A + N_B)$ and its statistical error
  $\sigma_A^2 = (1 - A^2)/(N_A + N_B)$.

In [ ]:
def inverse_variance_combine(
    values: list[float] | np.ndarray,
    errors: list[float] | np.ndarray,
) -> tuple[float, float]:
    """Inverse-variance-weighted combination of independent measurements.

    Parameters
    ----------
    values, errors
        Parallel arrays of central values and 1-sigma errors. Errors must be
        strictly positive.

    Returns
    -------
    (mean, error)
        Combined estimator and its propagated uncertainty.
    """
    v = np.asarray(values, dtype=float)
    e = np.asarray(errors, dtype=float)
    if v.shape != e.shape or v.ndim != 1:
        raise ValueError("values and errors must be 1-D arrays of the same shape")
    if not np.all(e > 0):
        raise ValueError("all errors must be strictly positive")
    w = 1.0 / e ** 2
    mean = float(np.sum(w * v) / np.sum(w))
    err  = float(np.sqrt(1.0 / np.sum(w)))
    return mean, err


def measurement_pull(
    v1: float, e1: float, v2: float, e2: float,
) -> float:
    """Standardised residual ``(v1 - v2) / sqrt(e1**2 + e2**2)``."""
    return float((v1 - v2) / np.sqrt(e1 ** 2 + e2 ** 2))


def asymmetry_from_counts(n_a: int, n_b: int) -> tuple[float, float]:
    """Binomial counting asymmetry ``(N_A - N_B)/(N_A + N_B)`` and its error."""
    n_total = n_a + n_b
    if n_total == 0:
        return float("nan"), float("nan")
    A = (n_a - n_b) / n_total
    sigma = float(np.sqrt(max(0.0, 1.0 - A ** 2) / n_total))
    return float(A), sigma


def sin2_theta_w_from_asymmetry(
    A: float, sigma_A: float,
) -> tuple[float, float]:
    r"""Convert :math:`A_\ell` (or :math:`A_{LR}`) to
    :math:`\sin^2\theta_W^{\mathrm{eff}}`.

    Uses :math:`A_\ell = 2 v_\ell a_\ell / (v_\ell^2 + a_\ell^2)` with
    :math:`v_\ell/a_\ell = 1 - 4\sin^2\theta_W^{\mathrm{eff}}`. The error is
    propagated linearly with slope :math:`d\sin^2\theta_W/dA \approx 1/8` in
    SLD's working range.
    """
    v_over_a = (1.0 - np.sqrt(1.0 - A ** 2)) / A
    sin2 = 0.25 * (1.0 - v_over_a)
    return float(sin2), float(sigma_A / 8.0)


def save_results_json(result: dict, name: str) -> Path:
    """Persist a result dict to ``MEASUREMENTS_DIR/{name}.json``."""
    out_path = MEASUREMENTS_DIR / f"{name}.json"
    with open(out_path, "w") as f:
        json.dump(result, f, indent=2, default=float)
    print(f"  Saved {out_path}")
    return out_path

## 4. Polarisation and year-group setup

The luminosity-weighted electron-beam polarisation enters both measurements:

$$ \langle \mathcal P_e \rangle \;=\; (1 + \xi)\,\frac{1}{N_Z}\sum_{i=1}^{N_Z}\mathcal P_i, $$

where $\mathcal P_i$ is the Compton-polarimeter measurement closest in time
to event $i$ and $\xi$ is a small chromaticity correction
(typically $\lesssim 10^{-3}$ at the SLC; we set $\xi = 0$). The error on
$\langle\mathcal P_e\rangle$ is taken from the `PHBM.dpol` field — more
conservative than the dispersion of per-event values, since the polarimeter
calibration uncertainty is correlated across each run.

### 4.1 Year-group structure

Following the published SLD analyses we pool **1997 and 1998** into a single
measurement and keep **1996** separate: the SLC polarised-electron source and
its operating point were essentially unchanged across 1997–98, while 1996
used the legacy strained-cathode source ($\langle\mathcal P_e\rangle \approx
0.76$) versus the upgraded source for 1997–98 ($\approx 0.73$).

In [ ]:
# Chromaticity correction (zero at the SLC sub-permille level).
XI_CHROMATICITY: float = 0.0

# Year-group structure shared by A_LR and the leptonic measurement.
YEAR_GROUPS: tuple[tuple[int, ...], ...] = (
    (1996,),
    (1997, 1998),
)

# 2001 leptonic preselection: year-dependent thrust-axis fiducial.
YEAR_FIDUCIAL: dict[int, float] = {
    1996: 0.80,
    1997: 0.90,
    1998: 0.90,
}

# Additive bias correction for the tau channel (2001 paper Table II): removes
# residual feed-through from tau-daughter charge mis-assignment. The
# corresponding corrections for A_e and A_mu are sub-permille and not applied.
YEAR_A_TAU_BIAS: dict[int, float] = {
    1996: -0.0182,
    1997: -0.0183,
    1998: -0.0183,
}


def get_polarisation(arr: ak.Array) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Per-event polarisation, its uncertainty, and the validity mask."""
    pol  = ak.to_numpy(ak.fill_none(
        ak.pad_none(arr.PHBM.pol,  target=1, axis=1), value=0.0)[:, 0])
    dpol = ak.to_numpy(ak.fill_none(
        ak.pad_none(arr.PHBM.dpol, target=1, axis=1), value=0.0)[:, 0])
    valid = np.abs(pol) > 0.1
    return pol, dpol, valid


def event_year(arr: ak.Array) -> np.ndarray:
    """Calendar year of each event from IEVENTH event timestamps."""
    evttime = ak.to_numpy(arr.IEVENTH.evttime)
    year = pd.to_datetime(evttime, unit='s').year.values.astype(np.int32)
    # Matches the package's 'event_year' quantity (sld_resurrect.event_view).
    return np.where(year == 1970, 1997, year)


def mean_polarisation(
    pol: np.ndarray, dpol: np.ndarray, mask: np.ndarray,
) -> tuple[float, float]:
    r"""Sample mean :math:`\langle |\mathcal P_e| \rangle` and its uncertainty.

    The error is the mean of ``|dpol|`` (rather than the dispersion of
    ``|pol|``), since polarimeter calibration errors are correlated across
    each run -- adding individual-event errors in quadrature would
    underestimate the systematic.
    """
    P_e  = float(np.mean(np.abs(pol[mask])))
    dP_e = float(np.mean(np.abs(dpol[mask])))
    return P_e, dP_e


def label_year_group(years: tuple[int, ...]) -> str:
    """Compact string label for a year-group: ``'1996'`` or ``'1997-1998'``."""
    return str(years[0]) if len(years) == 1 else f"{min(years)}-{max(years)}"


def group_unique_value(
    per_year: dict[int, float], group: tuple[int, ...], name: str,
) -> float:
    """Return the (single) value of ``per_year[y]`` for y in ``group``.

    Raises if any year is missing or if the years in the group disagree, since
    a year-group is treated as a single measurement and so must share the
    same fiducial / bias parameter.
    """
    missing = [y for y in group if y not in per_year]
    if missing:
        raise KeyError(f"Year(s) {missing!r} missing from the {name} table.")
    values = {per_year[y] for y in group}
    if len(values) > 1:
        raise ValueError(
            f"Years {group!r} have inconsistent {name} values: {sorted(values)!r}"
        )
    return values.pop()

## 5. Hadronic $A_{LR}$

For the hadronic event sample the raw left-right asymmetry is

$$ A_m \;=\; \frac{N_L - N_R}{N_L + N_R}, $$

and the **measured** left-right asymmetry at the run centre-of-mass energy is

$$ A_{LR}(E_{\mathrm{cm}}) \;=\; \frac{A_m}{\langle \mathcal P_e \rangle}. $$

The published 2000 SLD analysis converts this into the **pole asymmetry**
$A_{LR}^0 = A_{LR}(E_{\mathrm{cm}}) + \Delta_{EW}(E_{\mathrm{cm}})$, a small
additive correction that accounts for pure-photon exchange, $\gamma/Z$
interference, and initial-state radiation, computed run-by-run with the
ZFITTER 6.22 electroweak-fitting code. **ZFITTER is no longer available in a
form we can run on the released sample**, so this notebook reports the raw
measured $A_{LR}(E_{\mathrm{cm}})$ at the dataset-averaged $E_{\mathrm{cm}}$
without the pole correction (see §9). The resulting
$\sin^2\theta_W^{\mathrm{eff}}$ therefore differs slightly from the published
value by an amount consistent with the omitted shift, rather than a
discrepancy in the underlying data.

We compute $A_{LR}$ per year-group, then combine across year-groups by
inverse-variance weighting (the published combination uses a covariance
matrix that captures correlated polarisation systematics across years; we
use plain inverse-variance weighting on the per-year-group statistical
uncertainty, so the central value matches but the combined uncertainty is
slightly under-estimated relative to the published one).

In [ ]:
def measure_alr_year_group(
    arr: ak.Array,
    pol: np.ndarray,
    dpol: np.ndarray,
    year: np.ndarray,
    years_in_group: tuple[int, ...],
) -> dict | None:
    """Per-year-group A_LR from one (pre-selected) hadronic sample.

    Returns ``None`` if no valid-polarisation events from this group survive.
    Errors propagate the per-event statistical asymmetry and the mean
    polarisation uncertainty.
    """
    _, _, valid = get_polarisation(arr)
    mask = valid & np.isin(year, years_in_group)
    if not np.any(mask):
        return None

    pol_evt = pol[mask]
    n_left  = int(np.sum(pol_evt < 0))
    n_right = int(np.sum(pol_evt > 0))
    A_m, dA_m = asymmetry_from_counts(n_left, n_right)

    P_e, dP_e = mean_polarisation(pol, dpol, mask)
    P_e_corr  = P_e * (1.0 + XI_CHROMATICITY)

    A_LR  = A_m / P_e_corr
    dA_LR = abs(A_LR) * np.sqrt((dA_m / A_m) ** 2 + (dP_e / P_e_corr) ** 2)
    sin2, d_sin2 = sin2_theta_w_from_asymmetry(A_LR, dA_LR)

    return {
        'years':          list(years_in_group),
        'label':          label_year_group(years_in_group),
        'N_L':            n_left,
        'N_R':            n_right,
        'N_total':        n_left + n_right,
        'A_m':            A_m,
        'dA_m':           dA_m,
        'P_e':            P_e,
        'dP_e':           dP_e,
        'A_LR':           A_LR,
        'dA_LR':          dA_LR,
        'sin2_theta_W':   sin2,
        'd_sin2_theta_W': d_sin2,
    }


def measure_alr(
    arr: ak.Array,
    year_groups: tuple[tuple[int, ...], ...] = YEAR_GROUPS,
) -> dict:
    """Year-group-grained A_LR measurement plus its inverse-variance combination."""
    pol, dpol, _ = get_polarisation(arr)
    year = event_year(arr)

    by_group: dict[str, dict] = {}
    for years in year_groups:
        r = measure_alr_year_group(arr, pol, dpol, year, years)
        if r is not None:
            by_group[r['label']] = r

    if not by_group:
        nan = float('nan')
        combined = {'N_L': 0, 'N_R': 0, 'N_total': 0,
                    'A_LR': nan, 'dA_LR': nan,
                    'sin2_theta_W': nan, 'd_sin2_theta_W': nan}
    else:
        A_LR_avg, dA_LR_avg = inverse_variance_combine(
            [g['A_LR']  for g in by_group.values()],
            [g['dA_LR'] for g in by_group.values()],
        )
        sin2_avg, d_sin2_avg = sin2_theta_w_from_asymmetry(A_LR_avg, dA_LR_avg)
        combined = {
            'N_L':            sum(g['N_L']     for g in by_group.values()),
            'N_R':            sum(g['N_R']     for g in by_group.values()),
            'N_total':        sum(g['N_total'] for g in by_group.values()),
            'A_LR':           A_LR_avg,
            'dA_LR':          dA_LR_avg,
            'sin2_theta_W':   sin2_avg,
            'd_sin2_theta_W': d_sin2_avg,
        }

    return {'year_groups': by_group, 'combined': combined}


def print_alr_result(result: dict) -> None:
    """Print per-year-group then combined A_LR result."""
    print("--- Per-year-group hadronic A_LR ---------------------------------")
    for r in result['year_groups'].values():
        print(f"  [{r['label']}] N_L={r['N_L']:>6,}  N_R={r['N_R']:>6,}  "
              f"N_total={r['N_total']:>6,}")
        print(f"          A_m={r['A_m']:+.5f}±{r['dA_m']:.5f}  "
              f"<P_e>={r['P_e']:.4f}±{r['dP_e']:.4f}")
        print(f"          A_LR={r['A_LR']:.5f}±{r['dA_LR']:.5f}  "
              f"sin^2 theta_W={r['sin2_theta_W']:.5f}±{r['d_sin2_theta_W']:.5f}")

    c = result['combined']
    print("--- Combined (inverse-variance) ----------------------------------")
    print(f"  N_L={c['N_L']:,}  N_R={c['N_R']:,}  N_total={c['N_total']:,}")
    print(f"  A_LR (raw, no ZFITTER pole correction) = {c['A_LR']:.5f} ± {c['dA_LR']:.5f}")
    print(f"  sin^2 theta_W (eff)                    = {c['sin2_theta_W']:.5f} ± "
          f"{c['d_sin2_theta_W']:.5f}")


alr_result = measure_alr(selected['hadronic'])
print_alr_result(alr_result)
save_results_json(alr_result, 'alr')

## 6. Leptonic coupling asymmetries $A_e, A_\mu, A_\tau$

For each lepton channel SLD measures the **left-right forward-backward
asymmetry**

$$
\widetilde{A}_{FB}^\ell \;=\;
\frac{(N_F^L - N_B^L) - (N_F^R - N_B^R)}{N_L + N_R},
$$

where $L/R$ denotes the electron-beam helicity and $F/B$ is defined by the
charge-oriented thrust axis. At tree level

$$
\widetilde{A}_{FB}^\ell \;=\; \tfrac{3}{4}\,\langle \mathcal P_e \rangle\,A_\ell
\;\;\Longrightarrow\;\;
A_\ell \;=\; f_{\mathrm{geom}}(c)\,
\frac{\widetilde A_{FB}^\ell}{\langle \mathcal P_e \rangle},
$$

with the geometric correction $f_{\mathrm{geom}}(c) = (3 + c^2)/(3c)$ for a
$|\cos\theta_T| < c$ fiducial. The 2001 leptonic preselection uses a
year-dependent fiducial ($c = 0.8$ in 1996, $0.9$ in 1997–98), which is why
the year-group structure must be respected — years pooled into one
measurement share a single $f_{\mathrm{geom}}$.

### 6.1 Choice of $A_e$ extraction

Empirically, the cleanest extraction of $A_e$ at SLD comes from the **muon**
sample (the electron channel's counting-method $A_e$ is biased by Bhabha
$t$-channel exchange). This notebook follows that convention: $A_e$ is
reported from the muon-channel L/R counts, while $A_\mu, A_\tau$ each come
from their own L/R-FB asymmetry.

### 6.2 Tau bias correction

A small additive correction is applied to $A_\tau$ to remove residual
feed-through from $\tau$-daughter charge mis-assignment. The 2001 paper
quotes year-group-specific values; the equivalent corrections for $A_e$ and
$A_\mu$ are sub-permille and not applied here.

In [ ]:
def _geom_factor(cos_theta_max: float) -> float:
    """Geometric correction (3+c^2)/(3c) for a |cos theta| < c fiducial."""
    return (3.0 + cos_theta_max ** 2) / (3.0 * cos_theta_max)


def measure_leptonic_year_group(
    name: str,
    cos_theta: np.ndarray,
    pol: np.ndarray,
    dpol: np.ndarray,
    year: np.ndarray,
    years_in_group: tuple[int, ...],
    cos_theta_max: float,
    tau_bias: float,
) -> dict | None:
    """Per-(channel, year-group) L/R-FB extraction.

    Returns
    -------
    dict | None
        Per-group asymmetries (``A_LR`` from L/R counts; ``A_LRFB`` from
        L/R-FB) plus the derived A_e_from_channel and A_l. ``None`` if no
        valid-thrust, valid-polarisation events from this group survive.
    """
    valid_pol = np.abs(pol) > 0.1
    mask = valid_pol & ~np.isnan(cos_theta) & np.isin(year, years_in_group)
    if not np.any(mask):
        return None

    pol_evt = pol[mask]
    cos_evt = cos_theta[mask]

    is_left  = pol_evt < 0
    is_right = pol_evt > 0
    is_fwd   = cos_evt > 0
    is_bwd   = cos_evt < 0

    n_l  = int(np.sum(is_left))
    n_r  = int(np.sum(is_right))
    n_fl = int(np.sum(is_left  & is_fwd))
    n_bl = int(np.sum(is_left  & is_bwd))
    n_fr = int(np.sum(is_right & is_fwd))
    n_br = int(np.sum(is_right & is_bwd))
    n_total = n_l + n_r

    P_e, dP_e = mean_polarisation(pol, dpol, mask)
    geom = _geom_factor(cos_theta_max)

    # A_e from L/R counts only (no FB info -> no f_geom).
    A_LR, dA_LR = asymmetry_from_counts(n_l, n_r)
    A_e_channel = A_LR / P_e
    dA_e_channel = (1.0 / abs(P_e)) * np.sqrt(
        dA_LR ** 2 + (A_LR * dP_e / P_e) ** 2
    )

    # A_l from the L/R-FB asymmetry, with the f_geom correction and (for tau)
    # the additive bias correction.
    A_LRFB = (n_fl - n_bl - n_fr + n_br) / n_total
    dA_LRFB = float(np.sqrt(max(0.0, 1.0 - A_LRFB ** 2) / n_total))
    A_l = geom * A_LRFB / P_e
    if name == 'tau':
        A_l += tau_bias
    dA_l = (geom / abs(P_e)) * np.sqrt(
        dA_LRFB ** 2 + (A_LRFB * dP_e / P_e) ** 2
    )

    return {
        'years':            list(years_in_group),
        'label':            label_year_group(years_in_group),
        'cos_theta_max':    cos_theta_max,
        'f_geom':           geom,
        'tau_bias_applied': tau_bias if name == 'tau' else 0.0,
        'N_total':          n_total,
        'P_e':              P_e,
        'dP_e':             dP_e,
        'A_LR':             A_LR,
        'dA_LR':            dA_LR,
        'A_LRFB':           A_LRFB,
        'dA_LRFB':          dA_LRFB,
        'A_e_channel':      A_e_channel,
        'dA_e_channel':     dA_e_channel,
        'A_l':              A_l,
        'dA_l':             dA_l,
    }


def measure_leptonic_couplings(
    selected: dict[str, ak.Array],
    particles: dict[str, ak.Array],
    year_groups: tuple[tuple[int, ...], ...] = YEAR_GROUPS,
    year_fiducial: dict[int, float] = YEAR_FIDUCIAL,
    year_tau_bias: dict[int, float] = YEAR_A_TAU_BIAS,
) -> dict[str, dict]:
    """Measure A_e, A_mu, A_tau across the three lepton channels.

    For each (channel, year-group) pair we pool events from all years in the
    group, apply the group's fiducial-dependent f_geom (and the tau bias for
    A_tau), then combine across year-groups by inverse-variance weighting.
    """
    out: dict[str, dict] = {}
    for ch_short, ch_name in (('e', 'ee'), ('mu', 'mumu'), ('tau', 'tautau')):
        arr = selected[ch_name]
        sel = EventSelector.from_preset(f"leptonic_{ch_name}", arr, particles[ch_name])
        cos_theta = np.asarray(sel.get('thrust_vec_charged')[:, 2])

        pol, dpol, _ = get_polarisation(arr)
        year = event_year(arr)

        by_group: dict[str, dict] = {}
        for years in year_groups:
            r = measure_leptonic_year_group(
                name=ch_short,
                cos_theta=cos_theta,
                pol=pol, dpol=dpol, year=year,
                years_in_group=years,
                cos_theta_max=group_unique_value(year_fiducial, years, 'fiducial'),
                tau_bias=group_unique_value(year_tau_bias,  years, 'tau-bias'),
            )
            if r is not None:
                by_group[r['label']] = r

        # Combine year-groups for both A_l (final state) and A_e_channel
        # (initial state from this channel's L/R counts only).
        if by_group:
            A_l_avg, dA_l_avg = inverse_variance_combine(
                [g['A_l']  for g in by_group.values()],
                [g['dA_l'] for g in by_group.values()],
            )
            A_e_avg, dA_e_avg = inverse_variance_combine(
                [g['A_e_channel']  for g in by_group.values()],
                [g['dA_e_channel'] for g in by_group.values()],
            )
            combined = {
                'N_total':      sum(g['N_total'] for g in by_group.values()),
                'A_l':          A_l_avg, 'dA_l':          dA_l_avg,
                'A_e_channel':  A_e_avg, 'dA_e_channel':  dA_e_avg,
            }
        else:
            nan = float('nan')
            combined = {'N_total': 0,
                        'A_l': nan, 'dA_l': nan,
                        'A_e_channel': nan, 'dA_e_channel': nan}

        out[ch_short] = {'year_groups': by_group, 'combined': combined}
    return out


def summarise_leptonic(results: dict[str, dict]) -> dict[str, float]:
    """Print per-(channel, year-group), per-channel, and the universality
    average; return a flat summary dict."""
    print("--- Per-(channel, year-group) asymmetries ------------------------")
    for ch in ('e', 'mu', 'tau'):
        for r in results[ch]['year_groups'].values():
            print(
                f"  [{ch:<3} {r['label']:<9}] N={r['N_total']:<5,} | "
                f"P_e={r['P_e']:.4f} | "
                f"A_LRFB={r['A_LRFB']:+.4f} | "
                f"f_geom={r['f_geom']:.3f} | "
                f"A_{ch}={r['A_l']:+.4f}±{r['dA_l']:.4f}"
            )

    # A_e comes from the muon channel's L/R asymmetry (cleanest extraction).
    A_e,   dA_e   = (results['mu']['combined']['A_e_channel'],
                     results['mu']['combined']['dA_e_channel'])
    A_mu,  dA_mu  = (results['mu']['combined']['A_l'],
                     results['mu']['combined']['dA_l'])
    A_tau, dA_tau = (results['tau']['combined']['A_l'],
                     results['tau']['combined']['dA_l'])

    print("--- Per-channel combinations -------------------------------------")
    print(f"A_e (from muon L/R)         = {A_e:.4f} ± {dA_e:.4f}")
    print(f"A_mu                        = {A_mu:.4f} ± {dA_mu:.4f}")
    print(f"A_tau                       = {A_tau:.4f} ± {dA_tau:.4f}")

    # Lepton-universality combination.
    A_l, dA_l = inverse_variance_combine(
        [A_e, A_mu, A_tau], [dA_e, dA_mu, dA_tau],
    )
    sin2, d_sin2 = sin2_theta_w_from_asymmetry(A_l, dA_l)
    print("--- Universality average -----------------------------------------")
    print(f"Universal A_l        = {A_l:.4f} ± {dA_l:.4f}")
    print(f"sin^2 theta_W (lep.) = {sin2:.5f} ± {d_sin2:.5f}")

    return {
        'A_e':            A_e,   'dA_e':            dA_e,
        'A_mu':           A_mu,  'dA_mu':           dA_mu,
        'A_tau':          A_tau, 'dA_tau':          dA_tau,
        'A_l_universal':  A_l,   'dA_l_universal':  dA_l,
        'sin2_theta_W':   sin2,  'd_sin2_theta_W':  d_sin2,
    }


lep_results = measure_leptonic_couplings(selected, particles)
lep_summary = summarise_leptonic(lep_results)
save_results_json(
    {'summary': lep_summary, 'channels': lep_results},
    'leptonic_couplings',
)

## 7. Global fit of $\sin^2\theta_W^{\mathrm{eff}}$

Sections §5 and §6 produced two **independent** determinations of the
effective weak mixing angle:

| Channel set | Estimator | Sensitivity |
|---|---|---|
| Hadronic $Z\to q\bar q$ | $A_{LR}$ from L/R counting | electron-beam coupling $A_e$ via $A_{LR}^0 \approx A_e$ |
| Leptonic $Z\to\ell^+\ell^-$ | $\widetilde A_{FB}^\ell$ + universality | $A_\ell$ via the geometric correction |

Both map to $\sin^2\theta_W^{\mathrm{eff}}$ through the same tree-level
relation, the two samples are disjoint, and the (small) polarisation
systematic correlated across them is sub-dominant — so a plain
inverse-variance combination is the appropriate global estimator. The same
helper used to combine year-groups inside each block, `inverse_variance_combine`,
does the job here as well.

In [ ]:
def combine_alr_and_leptonic(alr_combined: dict, lep_summary: dict) -> dict:
    """Inverse-variance combine the hadronic A_LR and leptonic-fit estimators.

    Both inputs must carry ``sin2_theta_W`` and ``d_sin2_theta_W``.
    The pull is the standardised residual ``(alr - lep)/sqrt(de_alr^2 + de_lep^2)``.
    """
    s_alr = float(alr_combined['sin2_theta_W'])
    e_alr = float(alr_combined['d_sin2_theta_W'])
    s_lep = float(lep_summary['sin2_theta_W'])
    e_lep = float(lep_summary['d_sin2_theta_W'])

    s_global, e_global = inverse_variance_combine([s_alr, s_lep], [e_alr, e_lep])
    pull = measurement_pull(s_alr, e_alr, s_lep, e_lep)

    return {
        'sin2_alr':       s_alr, 'd_sin2_alr':       e_alr,
        'sin2_leptonic':  s_lep, 'd_sin2_leptonic':  e_lep,
        'sin2_global':    s_global, 'd_sin2_global': e_global,
        'pull':           pull,
    }


global_fit = combine_alr_and_leptonic(alr_result['combined'], lep_summary)

print('--- Inputs ------------------------------------------------------')
print(f"  A_LR (hadronic)    : {global_fit['sin2_alr']:.5f} ± {global_fit['d_sin2_alr']:.5f}")
print(f"  Leptonic fit       : {global_fit['sin2_leptonic']:.5f} ± {global_fit['d_sin2_leptonic']:.5f}")
print(f"  Pull (had - lep)/σ : {global_fit['pull']:+.2f}")
print('--- Combined ----------------------------------------------------')
print(f"  sin^2 theta_W^eff  : {global_fit['sin2_global']:.5f} ± {global_fit['d_sin2_global']:.5f}")

save_results_json(global_fit, 'sin2_theta_w_global')

## 8. Forest plot

A forest plot is the natural way to put the per-channel asymmetries, the two
per-block fits, the global combination, and the PDG reference *together*:
each measurement is one row carrying a central marker and a horizontal
uncertainty interval, and every row shares the same $\sin^2\theta_W$ axis.

Layout choices:

* **Two-block layout** separated by a horizontal rule — the aggregated fits
  (PDG, $A_{LR}$, $A_\ell$, combined) on top; the three individual lepton
  channels ($A_e, A_\mu, A_\tau$) below.
* **Highlight colour for the combined row** (firebrick red, filled square,
  bold tick label) — the same visual idiom as the headline row in PDG review
  tables.
* **Two translucent reference bands** track the two SLD inputs to the global
  fit (purple = $A_{LR}$, orange = $A_\ell$). The combined fit reads as the
  overlap region of the two bands, which is what an inverse-variance
  combination computes.
* **PDG 2025 collider average** as a faint dash-dot vertical line. A line
  rather than a band because the PDG uncertainty is much smaller than any
  SLD measurement here and a band would render as a hairline.
* **Inline value text** in math mode, right-aligned outside the plot area.
* **Distinct marker shapes per aggregated row** (pentagon / hexagon /
  triangle / square) for greyscale legibility; the three individual lepton
  channels share one circle marker.

### 8.1 Assemble the rows

In [ ]:
# PDG 2025 collider average for sin^2 theta_W^eff (collider average from the
# 'Electroweak model and constraints on new physics' chapter of the 2025 PDG
# Review of Particle Physics).
PDG_SIN2:     float = 0.23151
PDG_SIN2_ERR: float = 0.00016

# Per-lepton-channel sin^2 theta_W^eff (from each channel's own A_l value).
sin2_e,   d_sin2_e   = sin2_theta_w_from_asymmetry(lep_summary['A_e'],   lep_summary['dA_e'])
sin2_mu,  d_sin2_mu  = sin2_theta_w_from_asymmetry(lep_summary['A_mu'],  lep_summary['dA_mu'])
sin2_tau, d_sin2_tau = sin2_theta_w_from_asymmetry(lep_summary['A_tau'], lep_summary['dA_tau'])

# Order is top-to-bottom in the rendered plot.
forest_rows = [
    # ── Aggregated block ───────────────────────────────────────────
    ('pdg',          PDG_SIN2,                     PDG_SIN2_ERR),
    ('sld_alr',      global_fit['sin2_alr'],       global_fit['d_sin2_alr']),
    ('sld_leptonic', global_fit['sin2_leptonic'],  global_fit['d_sin2_leptonic']),
    ('sld_combined', global_fit['sin2_global'],    global_fit['d_sin2_global']),
    # ── Individual lepton channels ─────────────────────────────────
    ('A_tau',        sin2_tau, d_sin2_tau),
    ('A_mu',         sin2_mu,  d_sin2_mu),
    ('A_e',          sin2_e,   d_sin2_e),
]

forest_df = pd.DataFrame(forest_rows, columns=['target', 'value', 'unc']).set_index('target')
forest_df

### 8.2 Style and render

In [ ]:
import matplotlib.pyplot as plt
from quickstats.plots.forest_plot import ForestPlot

# ── Palette ─────────────────────────────────────────────────────
PALETTE = {
    'pdg':         '#2E7D5B',   # forest green   – PDG collider average
    'alr':         '#7B3F99',   # royal purple   – SLD A_LR (hadronic)
    'al':          '#E08319',   # warm amber     – SLD leptonic A_l
    'combined':    '#C0392B',   # firebrick      – this-work headline
    'channel':     '#3A6FB0',   # steel blue     – individual A_l markers
    'channel_lbl': '#283044',   # dark slate     – individual A_l labels
}

# ── Per-row label / marker / colour ─────────────────────────────
ROW_META = {
    'pdg':          {'label': r'PDG 2025 (collider average)',
                     'marker': 'p', 'color': PALETTE['pdg']},
    'sld_alr':      {'label': r'SLD $A_{LR}$ (this work)',
                     'marker': 'h', 'color': PALETTE['alr']},
    'sld_leptonic': {'label': r'SLD $A_{\ell}$ (this work)',
                     'marker': '^', 'color': PALETTE['al']},
    'sld_combined': {'label': r'SLD combined (this work)',
                     'marker': 's', 'color': PALETTE['combined']},
    'A_tau':        {'label': r'$A_\tau$',
                     'marker': 'o', 'color': PALETTE['channel']},
    'A_mu':         {'label': r'$A_\mu$',
                     'marker': 'o', 'color': PALETTE['channel']},
    'A_e':          {'label': r'$A_e$',
                     'marker': 'o', 'color': PALETTE['channel']},
}

label_map = {k: m['label'] for k, m in ROW_META.items()}

# color_map carries three dotted variants per key (plot/label/value); the
# individual lepton channels use a darker label colour than the marker.
color_map: dict = {}
for key, meta in ROW_META.items():
    color_map[f'{key}.plot']  = meta['color']
    color_map[f'{key}.label'] = meta['color']
    color_map[f'{key}.value'] = meta['color']
for key in ('A_e', 'A_mu', 'A_tau'):
    color_map[f'{key}.label'] = PALETTE['channel_lbl']
    color_map[f'{key}.value'] = PALETTE['channel_lbl']

# Per-row marker shape; combined-fit row also gets a thicker error bar.
styles_map: dict = {
    key: {
        'plot': {
            'marker':          meta['marker'],
            'markersize':      12 if key == 'sld_combined' else 10,
            'markeredgecolor': 'black',
            'markeredgewidth': 0.8,
        }
    }
    for key, meta in ROW_META.items()
}
styles_map['sld_combined']['errorbar'] = {
    'elinewidth': 2.4, 'capthick': 2.0, 'capsize': 6,
    'ecolor': PALETTE['combined'],
}

# ── Global styles ───────────────────────────────────────────────
styles = {
    'axis': {
        'labelsize': 17, 'major_width': 1.4, 'minor_width': 0.9,
        'x_axis_styles': {'direction': 'in',  'major_length': 8, 'minor_length': 4,
                          'top': True, 'bottom': True},
        'y_axis_styles': {'direction': 'out', 'major_length': 6, 'minor_length': 0,
                          'left': True, 'right': False},
    },
    'grid': {
        'axis': 'x', 'which': 'major', 'linestyle': ':', 'linewidth': 0.5,
        'color': '#BBBBBB', 'zorder': 0,
    },
    'plot': {
        'marker': 'o', 'markersize': 10, 'linestyle': 'None',
        'markeredgecolor': 'black', 'markeredgewidth': 0.8, 'zorder': 5,
    },
    'errorbar': {
        'linestyle': 'none', 'linewidth': 0,
        'elinewidth': 1.6, 'capsize': 5, 'capthick': 1.5, 'zorder': 4,
    },
    'hline': {'linestyle': '--', 'linewidth': 0.9, 'color': '#888888', 'zorder': 1},
    'value.text': {
        'fontsize': 16, 'verticalalignment': 'center',
        'horizontalalignment': 'left', 'clip_on': False,
    },
}

config = {
    'error_artist': 'errorbar',
    'show_values':  True,
    'value_xpos':   0.75,
    'value_fmt':    r'${value:.5f} \pm {upper_unc_1:.5f}$',
    'draw_legend':  False,
    'draw_grid':    True,
}

plot = ForestPlot(
    forest_df,
    color_map=color_map, label_map=label_map,
    styles=styles, styles_map=styles_map, config=config,
)

ax = plot.draw(
    nominal_column='value', error_columns='unc',
    # Separator below the combined-fit row: rows 0-3 aggregated,
    # rows 4-6 individual channels.
    line_indices=[4],
    xlabel=r'$\sin^2\theta_W^{\mathrm{eff}}$',
    xmin=0.229, xmax=0.2415,
    ypad=(0.10, 0.10),
    row_spacing=1.25,
)

# ── Reference bands and PDG line (drawn after the forest-plot artists) ──
ax.axvspan(
    global_fit['sin2_alr']      - global_fit['d_sin2_alr'],
    global_fit['sin2_alr']      + global_fit['d_sin2_alr'],
    color=PALETTE['alr'], alpha=0.16, zorder=0.5, linewidth=0,
)
ax.axvspan(
    global_fit['sin2_leptonic'] - global_fit['d_sin2_leptonic'],
    global_fit['sin2_leptonic'] + global_fit['d_sin2_leptonic'],
    color=PALETTE['al'],  alpha=0.16, zorder=0.5, linewidth=0,
)
ax.axvline(
    PDG_SIN2, color=PALETTE['pdg'],
    linestyle=(0, (5, 2, 1, 2)), linewidth=1.2, alpha=0.85, zorder=1,
)

# Bold the combined-fit ytick label so the headline row reads as the headline.
combined_label = ROW_META['sld_combined']['label']
for tick in ax.get_yticklabels():
    if tick.get_text() == combined_label:
        tick.set_fontweight('bold')
        break

outfile = PLOTS_DIR / 'sin2_theta_w.pdf'
ax.figure.savefig(outfile, bbox_inches='tight')
print(f'Forest plot saved to {outfile}')
plt.show()

## 9. Scientific limitations

The numerical results above are not a re-derivation of the published SLD
measurements. Two specific limitations are worth highlighting; both are
described in the dataset paper and are reproduced here so a reader can
interpret the numbers without leaving the notebook.

### 9.1 Hadronic $A_{LR}$ — missing pole-asymmetry correction

Our per-year-group hadronic event counts (and their $L/R$ splits) and the
resulting raw $A_{LR}$ values track the corresponding numbers in the 2000
high-precision SLD analysis ([hep-ex/0004026](https://arxiv.org/abs/hep-ex/0004026))
very closely. The residual differences are at the percent level on the
counts and smaller than the polarisation uncertainty on the asymmetry.

To compare with the Standard-Model prediction, the published analysis
converts the measured $A_{LR}(E_{\mathrm{cm}})$ into the **pole asymmetry**

$$ A_{LR}^0 \;=\; A_{LR}(E_{\mathrm{cm}}) \;+\; \Delta_{EW}(E_{\mathrm{cm}}), $$

a small additive correction that accounts for **pure-photon exchange**,
**$\gamma/Z$ interference**, and **initial-state radiation**, computed
run-by-run with the dedicated electroweak-fitting code ZFITTER 6.22. ZFITTER
is no longer available in a form we can run on the released sample, so this
notebook reports the raw, measured $A_{LR}$ at the dataset-averaged
$E_{\mathrm{cm}}$ without this correction. The resulting
$\sin^2\theta_W^{\mathrm{eff}}$ therefore differs slightly from the
published value by an amount consistent with the omitted shift, rather than
a discrepancy in the underlying data.

### 9.2 Leptonic asymmetries — selection and extraction differences

Two effects complicate a direct numerical comparison with the 2001
leptonic-coupling paper
([hep-ex/0010015](https://arxiv.org/abs/hep-ex/0010015)).

**Selected event counts differ from the published numbers.** After applying
our reconstruction of the published preselection and channel cuts (see
notebook 01), our selected counts in the dilepton channels differ from
those reported in 2001, most visibly in $\mu\mu$ and $\tau\tau$. The
original selection software is not available with the released dataset, so
the precise origin of this difference cannot be pinned down here. The most
likely cause is a residual difference in **track-quality**,
**particle-identification**, or **fiducial requirements** that the public
documentation does not fully specify.

**Counting method vs. maximum-likelihood fit.** The published analysis uses
an unbinned maximum-likelihood fit to the polarised differential cross
section, with **per-event acceptance corrections** folded into the
likelihood and **$A_e$ and $A_\ell$ floated simultaneously**:

$$ \frac{1}{\sigma}\,\frac{d\sigma}{d\cos\theta} \;\propto\;
   (1 + \cos^2\theta)\,(1 - \mathcal P_e A_e)
   \;+\; 2\cos\theta\,(A_e - \mathcal P_e)\,A_\ell . $$

This notebook uses direct event counting with the analytic fiducial
correction $f_{\mathrm{geom}}(c) = (3 + c^2)/(3c)$ and takes $A_e$ from the
muon channel's $L/R$ asymmetry. Acceptance variation with $|\cos\theta_T|$
is therefore not corrected event-by-event in our extraction, and the
$A_e$–$A_\ell$ off-diagonal correlation that the joint fit captures is not
propagated.

Together with the omission of all non-statistical systematic uncertainties,
these two effects account for the modest pull of our universal $A_\ell$
relative to the 2001 published value.

### 9.3 Why this is still useful

Despite these caveats, the close per-year-group agreement of the hadronic
counts and raw $A_{LR}$, the recovery of the expected $L/R$ splitting in
the three leptonic $\cos\theta$ distributions
(notebook 02), and the consistency of the independently extracted
$\sin^2\theta_W^{\mathrm{eff}}$ values across the hadronic and leptonic
channels demonstrate that the translated dataset preserves the
polarisation-sensitive content of the SLD reconstruction at the level
required for downstream physics and machine-learning use, and is a faithful
starting point for analyses that supply the missing radiative corrections
and detector-level systematic treatment.